XI coefficient-count update (2026-09-06). Counts use the tested module without decimal rounding, and an empty expansion has zero terms. Outputs were cleared because old counts are not reproducible under the correction. Other historical exploratory code is not a validated research result; see `RESEARCH_VALIDATION.md`.


In [ ]:
import numpy as np
import itertools
import qiskit.quantum_info as qi 
import time
import time
import csv
import sys
import matplotlib.pyplot as plt



# produce all binary strings of length n with k 1s. If k is None then all possible binary strings of length n produced
def get_binary_strings(n, k=None) -> list:
    '''
    produce all binary strings of length n with k 1s
    returns list with binary lists 
    '''
    final = []
    def kbits(r):
        result = []
        for bits in itertools.combinations(range(n), r):
            s = [0] * n
            for bit in bits:
                s[bit] = 1
            result.append(s)   
        return result

    if k != None:
        return kbits(k)
    
    for i in range(n + 1):
        final = final + kbits(i)
        
    return final

def get_circuit_operators( lst_x=None, lst_z=None):
    '''
    note the order! (x,z)
    '''
    return  qi.Pauli((lst_z,lst_x)).to_matrix()




def generate_QAOA_operator(nqubits, locality, number_of_terms, number_of_layers=1, beta_angle=None, gamma_angle=None):
    from QAOA_locality import generate_random_QAOA_operator
    beta = None if beta_angle is None or len(beta_angle) == 0 else beta_angle
    gamma = None if gamma_angle is None or len(gamma_angle) == 0 else gamma_angle
    return generate_random_QAOA_operator(nqubits, locality, number_of_terms, number_of_layers, beta, gamma)
   
   




def count_solutions_XI(nqubits, Operator, tol=1e-10, if_print=False):
    from QAOA_locality import count_solutions_XI as count_exact
    if np.shape(Operator) != (2**nqubits, 2**nqubits):
        raise ValueError("Operator size does not match nqubits.")
    result = count_exact(Operator, tol)
    if if_print:
        print("XI count, maximum locality, average locality:", result[3], result[1], result[2])
    return result



def make_grid(nqubits: int, max_loc: int, max_terms: int, number_of_layers=1, beta_angle=[], gamma_angle=[])->np.ndarray:
    grid = np.zeros((max_loc,max_terms,3))
    for i in range(1,max_loc+1):
        for j in range(1,max_terms+1):
            if i==nqubits and j>1:
                continue
            ans = generate_QAOA_operator(nqubits=nqubits,locality=i,number_of_terms=j, number_of_layers=number_of_layers,beta_angle=beta_angle,gamma_angle=gamma_angle)
            ans, max_loc, avg_loc, len_ans = count_solutions_XI(nqubits,ans,1e-10)
            grid[i-1][j-1][0] = max_loc
            grid[i-1][j-1][1] = avg_loc
            grid[i-1][j-1][2] = len_ans                
    return grid

def plot_grid(grid: np.ndarray, param_to_show=2):

    localities = [i + 1 for i in range(grid.shape[0])]
    num_of_terms = [i + 1 for i in range(grid.shape[1])]

    param_to_show_name = ''
    if param_to_show == 2:
        param_to_show_name = 'total terms'
    elif param_to_show == 1:
        param_to_show_name = 'average locality'
    else:
        param_to_show_name = 'max locality'


    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(grid[:,:,param_to_show], interpolation='nearest', cmap='plasma')
    
    # ax.set_yticks(localities)
    # ax.set_xticks(num_of_terms)

    fig.colorbar(cax)
    
    for locality in localities:
        for number_of_terms in num_of_terms:
            # print(locality,number_of_terms)
            ax.text(number_of_terms-1, locality-1, '{:.2f}'.format(grid[locality-1][number_of_terms-1][param_to_show]),  ha='center', va='center')
            
    ax.set_xlabel('No. of Terms', fontsize = 15)
    ax.set_ylabel('Locality', fontsize = 15)
    ax.set_title('Qubits {}, {}'.format(len(localities), param_to_show_name), fontsize = 20)

    plt.show()



In [ ]:
ans = generate_QAOA_operator(3,2,2)
ans,loc,avg, len_ans = count_solutions_XI(3,ans,1e-10)
print(ans)

In [ ]:
grid = make_grid(5, 5, 5)

In [ ]:
plot_grid(grid,1)

In [ ]:
# %load_ext line_profiler

In [ ]:
# %lprun -f make_grid make_grid(5,5,5)

In [ ]:
# %lprun -f generate_QAOA_operator generate_QAOA_operator(5,3,3)